# Lecture 24: Analyzing Categorical Outcomes

**Ling 250/450 — Data Science for Linguistics**

So far in our statistics unit, most of the methods we've used assume a **continuous** response variable: reaction times, formant values, acoustic measurements, ratings on a numeric scale. But a huge amount of linguistic data is **categorical**: which variant a speaker used, whether a listener identified a sound as /p/ or /b/, whether a word appeared in a given construction or not.

This lecture introduces the **chi-squared test**, the workhorse method for categorical-vs-categorical questions, and then circles back to **logistic regression** (which we first saw in Lecture 22) to show how the two are really answering the same kind of question with different machinery.

## Overview

1. **Goodness-of-fit chi-squared**: does an observed distribution match an expected one?
2. **Chi-squared test of independence**: are two categorical variables associated?
3. **Fisher's exact test**: the small-sample alternative.
4. **From contingency tables to logistic regression**: the same question, framed as a model.
5. **When to reach for logistic regression instead**.

Everything in this notebook uses base R — no new packages required.

---

## Part 1: Goodness-of-Fit

### The question

Suppose you have counts in a set of categories, and a theoretical expectation about how those counts *should* be distributed. A goodness-of-fit chi-squared test asks: **is the observed distribution consistent with the expected one, or does it depart from it more than chance would predict?**

### A linguistic example

English has well-studied letter and phoneme frequencies. Suppose a student claims to have collected a 1,000-word sample of "naturalistic" English text, and you want to sanity-check whether the distribution of vowels looks plausible. You count occurrences of the five vowel letters and compare against reference proportions from a large corpus.

In [ ]:
# Observed vowel counts in the student's 1,000-word sample
observed_counts <- c(a = 310, e = 455, i = 260, o = 280, u = 95)

# Reference proportions from a large English corpus (illustrative)
reference_proportions <- c(a = 0.26, e = 0.39, i = 0.18, o = 0.12, u = 0.05)

# Sanity check: proportions should sum to 1
sum(reference_proportions)

R's `chisq.test()` accepts an observed count vector and a `p` argument giving the expected proportions:

In [ ]:
gof_result <- chisq.test(x = observed_counts, p = reference_proportions)
gof_result

### Interpreting the output

Three things to notice:

- **X-squared**: the test statistic, which measures how far the observed counts are from the expected counts (squared and rescaled so that bigger = more discrepancy).
- **df**: degrees of freedom. For goodness-of-fit with $k$ categories, $df = k - 1$.
- **p-value**: probability of seeing a discrepancy this large (or larger) if the observed data really did come from the reference distribution.

We can inspect the expected counts and the residuals directly:

In [ ]:
gof_result$expected

# Pearson residuals: (observed - expected) / sqrt(expected)
# Values with absolute magnitude > ~2 flag categories that contribute
# disproportionately to the overall discrepancy.
gof_result$residuals

Residuals are often more interesting than the p-value. A significant chi-squared tells you *something* is off, but the residuals tell you *which categories* are driving the discrepancy.

---

## Part 2: Test of Independence

### The question

Now suppose you have **two** categorical variables and want to know whether they're associated. This is the more common use of chi-squared in linguistics.

Classic examples:

- Does the choice of a sociolinguistic variant (e.g., *-ing* vs. *-in'*) depend on speaker gender?
- Is a particular construction more common in one register than another?
- Do two words collocate more often than chance would predict?

### A worked example: (ING) variation

We'll use a made-up but realistic dataset: observations of word-final *-ing* tokens coded as either the standard variant (`ing`) or the non-standard variant (`in`), split by speaker gender.

In [ ]:
ing_table <- matrix(
  c(120,  80,
    105, 195),
  nrow = 2,
  byrow = TRUE,
  dimnames = list(
    gender  = c("female", "male"),
    variant = c("ing",    "in")
  )
)
ing_table

Before running a test, look at the marginal proportions — chi-squared can tell you the association is non-random, but you still want to describe the direction.

In [ ]:
# Row proportions: proportion of each variant within each gender
prop.table(ing_table, margin = 1)

Females use the standard variant 60% of the time in this sample, males 35%. Note also that the row totals are no longer equal — there are 200 female tokens and 300 male tokens — so the expected counts will not simply be column averages. The chi-squared test asks whether the gap in proportions is large enough to be unlikely under the null hypothesis of no association.

In [ ]:
independence_result <- chisq.test(ing_table)
independence_result

### Degrees of freedom

For an $r \times c$ contingency table, $df = (r - 1)(c - 1)$. A 2×2 table therefore has $df = 1$. Larger tables (say, variant × dialect region with 4 regions) give more df.

### Yates's continuity correction

By default, `chisq.test()` applies **Yates's continuity correction** to 2×2 tables. This is a slightly conservative adjustment intended to improve the chi-squared approximation for small samples. You can turn it off with `correct = FALSE`:

In [ ]:
chisq.test(ing_table, correct = FALSE)

For our sample size the difference is small. For very small samples, the correction matters more — but at that point you should really be using Fisher's exact test (next).

### Where does X-squared come from?

`chisq.test()` is a black box until you've computed the statistic once by hand. The formula is:

$$X^2 = \sum_{i} \frac{(O_i - E_i)^2}{E_i}$$

where $O_i$ is the observed count in cell $i$, $E_i$ is the expected count in that cell under the null hypothesis of independence, and the sum runs over every cell of the contingency table.

The trick is computing $E_i$. Under the null hypothesis that row and column variables are independent, the expected count in a cell is the product of its row total and column total, divided by the grand total:

$$E_{ij} = \frac{(\text{row } i \text{ total}) \times (\text{column } j \text{ total})}{\text{grand total}}$$

Let's do this explicitly on `ing_table`.

In [ ]:
# Row, column, and grand totals
row_totals   <- rowSums(ing_table)
col_totals   <- colSums(ing_table)
grand_total  <- sum(ing_table)

row_totals
col_totals
grand_total

In [ ]:
# Expected count in each cell, assuming independence:
# outer() multiplies each row total by each column total
expected_table <- outer(row_totals, col_totals) / grand_total
expected_table

Notice that every row of `expected_table` has the *same* ratio of `ing` to `in` — that's what independence means: if the variant didn't depend on gender, both genders would show the overall marginal split of the variant.

In [ ]:
# Apply the chi-squared formula cell by cell, then sum
cell_contributions <- (ing_table - expected_table)^2 / expected_table
cell_contributions

x_squared_by_hand <- sum(cell_contributions)
x_squared_by_hand

Compare to what `chisq.test()` reports (use `correct = FALSE` to match — Yates's correction adjusts each $|O - E|$ downward by 0.5 before squaring, so it won't match our by-hand version unless we disable it):

In [ ]:
chisq.test(ing_table, correct = FALSE)$statistic

Same number. Each cell's contribution — the entries of `cell_contributions` above — tells you which part of the table is most responsible for the overall discrepancy. That's the same information the Pearson residuals we saw earlier give you, just not yet divided through by $\sqrt{E_i}$.

### An aside: chi-squared is symmetric

Look back at how we built `expected_table`: row totals times column totals, divided by the grand total. Nothing in that formula distinguishes rows from columns. If we transposed `ing_table` and put `variant` in the rows and `gender` in the columns, we'd get exactly the same expected counts, the same cell-by-cell contributions, and the same $X^2$. The null hypothesis $P(\text{gender}, \text{variant}) = P(\text{gender}) \cdot P(\text{variant})$ is itself symmetric — it doesn't privilege one variable as "the outcome."

There are actually three different sampling schemes that can produce a 2×2 table like this one:

1. **Neither margin fixed in advance** — you transcribed $N$ tokens of `-ing` from a corpus and cross-classified each by gender and variant. Neither row totals nor column totals were set by the researcher. This is a test of *independence* in the strict sense.
2. **One margin fixed by design** — you deliberately collected 200 female tokens and 300 male tokens. Now the row totals aren't random; they were chosen. You're really asking whether the distribution of variants is *the same* across the two groups — a test of *homogeneity of proportions*. The arithmetic and the p-value are unchanged, but you'd naturally describe gender as a grouping variable rather than as an outcome.
3. **Both margins fixed** — rare in linguistics, but this is the setup for Fisher's original tea-tasting experiment and the motivation for Fisher's exact test.

The upshot: in a chi-squared framework there is no intrinsic independent or dependent variable — just two categorical variables and a question about whether they're associated. That is a real contrast with **regression**, which is fundamentally asymmetric. `glm(variant ~ gender)` models $P(\text{variant} \mid \text{gender})$, and `glm(gender ~ variant)` is a different model with different coefficients. That asymmetry is exactly what lets regression generalize naturally to mixed predictor types, predictions for new observations, and (with care) causal interpretation — things chi-squared cannot do.

---

## Part 3: Fisher's Exact Test

Chi-squared is an **approximation** — it relies on the test statistic being approximately chi-squared distributed, which only holds when expected counts are reasonably large. A common rule of thumb: all expected cell counts should be at least 5.

When that rule is violated (small samples, rare categories), use **Fisher's exact test**, which computes the probability of the observed table directly from the hypergeometric distribution — no approximation required.

In [ ]:
# A tiny hypothetical dataset: did 14 bilingual children prefer
# SVO or SOV word order in an elicited production task,
# split by dominant language?
small_table <- matrix(
  c(6, 1,
    2, 5),
  nrow = 2,
  byrow = TRUE,
  dimnames = list(
    dominant = c("English", "Japanese"),
    order = c("SVO", "SOV")
  )
)
small_table

fisher.test(small_table)

Fisher's exact test also gives you an **odds ratio** — a measure of effect size we'll see again in a moment.

---

## Part 4: From Contingency Tables to Logistic Regression

Here's the conceptual bridge that ties this lecture back to Lecture 22.

A 2×2 chi-squared test of independence asks: **does the probability of outcome Y depend on group X?** That's exactly the question logistic regression answers when you have one binary predictor and one binary outcome. They are different pieces of machinery for the *same* question.

Let's demonstrate by re-analyzing the (ING) data as a logistic regression. First we need it in **long format** — one row per observation — rather than a contingency table.

In [ ]:
# Expand the contingency table into a long data frame
ing_long <- data.frame(
  gender  = c(rep("female", 200), rep("male", 300)),
  variant = c(rep("ing", 120), rep("in",  80),
              rep("ing", 105), rep("in", 195))
)

# Code the outcome as a binary 0/1 variable:
# 1 = standard variant ("ing"), 0 = non-standard ("in")
ing_long$is_standard <- as.integer(ing_long$variant == "ing")

head(ing_long)
table(ing_long$gender, ing_long$variant)

Now fit a logistic regression with gender as the sole predictor:

In [ ]:
mod_ing <- glm(
  is_standard ~ gender,
  data = ing_long,
  family = binomial
)
summary(mod_ing)

### Reading the output as a chi-squared analogue

- The **intercept** corresponds to the log-odds of the standard variant for the reference group (females, because `female` comes before `male` alphabetically).
- The **`gendermale`** coefficient is how much the log-odds change when we move from female to male speakers. It's negative, meaning males are *less* likely to use the standard variant — consistent with the row proportions we saw earlier.
- The Wald **z-test** p-value on that coefficient answers essentially the same question as the chi-squared test of independence.

Compare:

In [ ]:
# p-value from the chi-squared test (without Yates's correction,
# for a fairer comparison)
chisq.test(ing_table, correct = FALSE)$p.value

# p-value from the logistic regression coefficient
coef(summary(mod_ing))["gendermale", "Pr(>|z|)"]

These are not numerically identical — chi-squared and the Wald z-test use slightly different approximations — but they lead to the same conclusion. For a 2×2 table, they are effectively interchangeable ways of asking "are these two variables associated?"

### Odds ratios

Exponentiating a logistic regression coefficient gives an **odds ratio**: the multiplicative change in the odds of the outcome associated with a one-unit change in the predictor.

In [ ]:
# Odds of the standard variant for males, divided by odds for females
exp(coef(mod_ing)["gendermale"])

An odds ratio below 1 means the outcome is less likely in the comparison group; above 1 means more likely. This is a much more interpretable effect size than `X-squared`, which has no direct real-world meaning.

---

## Part 5: When to Reach for Logistic Regression Instead

If chi-squared and logistic regression answer the same question for a 2×2 table, when should you prefer one over the other?

**Use chi-squared when:**

- You're doing quick exploratory analysis of a contingency table.
- Both variables are categorical and you have no continuous predictors to add.
- You want a single simple statement about association.

**Use logistic regression when:**

- You want to include **multiple predictors** at once (e.g., gender *and* age *and* formality).
- You want to include **continuous predictors** (e.g., speaker age in years, word frequency).
- You care about **effect sizes**, not just significance — odds ratios are much more informative than chi-squared statistics.
- You want to generate **predictions** for new data.
- You need to account for **random effects** (e.g., by-speaker or by-item variation — mixed-effects logistic regression, `glmer()` from **lme4**).

In modern linguistics research papers, you'll see logistic regression far more often than chi-squared, precisely because real research questions almost always involve more than two variables. But chi-squared remains useful as a quick first look and as a teaching tool for building intuition about categorical association.

---

## Challenge Exercise

<details>
<summary><b>Challenge:</b> Extending the (ING) analysis</summary>

Imagine your dataset also includes speech style, coded as `casual` or `formal`. Here are the counts:

```
                 ing   in
female casual    55    85
female formal    87    23
male   casual    32   110
male   formal    63    45
```

1. Build the long-format data frame for this extended dataset.
2. Fit a logistic regression with `gender` **and** `style` as predictors. Which has the larger effect on the log-odds of the standard variant?
3. Try adding an **interaction** (`gender * style`). Is there evidence that the effect of style differs by gender?
4. Why would you *not* be able to answer question 3 with a simple chi-squared test?

<details>
<summary>Hint for (1)</summary>

You can build the data frame using repeated calls to `rep()`, the same way we did for `ing_long` above — just with four groups instead of two. Alternatively, construct a 3D array and use `as.data.frame()`.
</details>

<details>
<summary>Answer for (4)</summary>

Chi-squared tests association between **two** categorical variables at a time. With three variables, you'd have to do multiple pairwise tests, which (a) inflates your Type I error and (b) cannot detect interactions — the idea that the effect of one variable *depends on the value of another*. Regression handles this naturally because it models each predictor's contribution while controlling for the others.
</details>

</details>

---

## Quick Reference

### Chi-squared tests in R

| Task | R code |
|------|--------|
| Goodness of fit | `chisq.test(x = counts, p = expected_proportions)` |
| Test of independence | `chisq.test(contingency_table)` |
| Disable Yates's correction | `chisq.test(tbl, correct = FALSE)` |
| Inspect expected counts | `result$expected` |
| Inspect residuals | `result$residuals` |
| Small-sample alternative | `fisher.test(contingency_table)` |

### Logistic regression (review from Lecture 22)

| Task | R code |
|------|--------|
| Fit binary logistic | `glm(y ~ x, data = d, family = binomial)` |
| Coefficient summary | `summary(mod)` |
| Odds ratios | `exp(coef(mod))` |
| Predicted probabilities | `predict(mod, type = "response")` |
| Mixed-effects version | `lme4::glmer(y ~ x + (1 | speaker), data = d, family = binomial)` |

### Which test when?

| Situation | Recommended tool |
|-----------|-------------------|
| 2×2 table, moderate sample | Chi-squared or logistic regression |
| 2×2 table, tiny sample or rare cells | Fisher's exact |
| Multi-category × multi-category table | Chi-squared of independence |
| Multiple predictors, possibly continuous | Logistic regression |
| Repeated measures per speaker/item | Mixed-effects logistic regression |

### Further reading

- Levshina, Natalia. 2015. *How to Do Linguistics with R*. John Benjamins. Chapters 9 and 12 cover categorical data analysis in depth.
- Winter, Bodo. 2019. *Statistics for Linguists: An Introduction Using R*. Routledge. Chapter 12 on logistic regression is especially approachable.
- Agresti, Alan. 2018. *An Introduction to Categorical Data Analysis* (3rd ed.). Wiley. The standard reference — not linguistics-specific, but thorough.